# Konoros-Sec v1.1 — Colab Train & Test (defensive only)
Repo đã có sẵn data thật: `data/raw/so_all.jsonl` (208 Q&A: python/js/linux/security/networking), `so_python.jsonl`, `so_security.jsonl`, `web_docs.txt`. Chạy lần lượt các cell dưới đây trên GPU Colab (T4 miễn phí đủ cho tiny; small cần bật grad_checkpoint). Quota SE API reset mỗi ngày — muốn lấy thêm thì chạy lại `fetch_all.py` trên Colab hoặc gắn API key miễn phí (stackapps.com).

In [ ]:
!pip -q install torch numpy pyyaml tqdm tensorboard pytest requests datasets
!python -m pytest test/ -q  # kỳ vọng: pass hết (gồm test GQA + KV-cache)

In [ ]:
# 1. Build .bin từ data THẬT (SO python + security + docs). Muốn thêm data: chạy collector rồi thêm file vào --input
!python scripts/prepare_data.py --input data/raw/so_all.jsonl,data/raw/web_docs.txt,data/raw/train.txt --train-out data/processed/train.bin --val-out data/processed/val.bin --tok-out data/tokenizer/byte.json

In [ ]:
# 2a. Pretrain TINY smoke test (~3M params, vài phút trên T4)
!python -m training.train --config config/model/tiny.yaml
# 2b. Pretrain SMALL (~30M, GQA 8q/2kv, ctx 1024) — chạy khi tiny đã loss giảm
# !python -m training.train --config config/model/small.yaml

In [ ]:
!ls experiments/v0.1/checkpoints/
!python -m evaluation.evaluate --config config/model/tiny.yaml --ckpt experiments/v0.1/checkpoints/step_005000.pt

In [ ]:
# 3. SFT v1.1 defensive
!python -m training.sft --config config/model/tiny.yaml --data data/sft/security_sft.jsonl --base-ckpt experiments/v0.1/checkpoints/step_005000.pt --out experiments/v1.1/sft.pt --max-steps 60 --lr 1e-5

In [ ]:
!python -m security.eval.safety_eval --config config/model/tiny.yaml --ckpt experiments/v1.1/sft.pt
!python -m inference.generate --ckpt experiments/v1.1/sft.pt --prompt "<user>How do I secure my home WiFi?</user>" --max-new-tokens 80 --temperature 0.0

### Lấy thêm data (chạy trên Colab, tuân thủ API + CC-BY-SA)
`!python data_collectors/stackoverflow/collect_so.py --tag python --pages 5 --out data/raw/so_python2.jsonl` — rồi thêm file vào `--input` ở cell 1 và build lại `.bin`.